### HuggingFaceEndpointEmbeddings

In [2]:
import os

from dotenv import load_dotenv
from langchain_teddynote import logging


load_dotenv()
logging.langsmith("test0914")

print("OpenAI 키 로드됨 : ", bool(os.getenv("OPENAI_API_KEY")))
print("LangSmith 키 로드됨 : ", bool(os.getenv("LANGSMITH_API_KEY")))
print("LangSmith 프로젝트 : ", os.getenv("LANGSMITH_PROJECT"))
print("HF_TOKEN 키 로드됨 : ", bool(os.getenv("HF_TOKEN_API_KEY")))

LangSmith 추적을 시작합니다.
[프로젝트명]
test0914
OpenAI 키 로드됨 :  True
LangSmith 키 로드됨 :  True
LangSmith 프로젝트 :  test0914
HF_TOKEN 키 로드됨 :  True


In [3]:
import warnings

warnings.filterwarnings("ignore")

os.environ["HF_HOME"] = "./cache/"

In [4]:
texts = [
    "안녕, 만나서 반가워.",
    "LangChain simplifies the process of building applications with large language models",
    "랭체인 한국어 튜토리얼은 LangChain의 공식 문서, cookbook 및 다양한 실용 예제를 바탕으로 하여 사용자가 LangChain을 더 쉽고 효과적으로 활용할 수 있도록 구성되어 있습니다. ",
    "LangChain은 초거대 언어모델로 애플리케이션을 구축하는 과정을 단순화합니다.",
    "Retrieval-Augmented Generation (RAG) is an effective technique for improving AI responses.",
]

In [7]:
from langchain_huggingface.embeddings import HuggingFaceEndpointEmbeddings

model_name = "intfloat/multilingual-e5-large-instruct"

hf_embeddings = HuggingFaceEndpointEmbeddings(
    model= model_name,
    task= "feature-extraction",
    huggingfacehub_api_token= os.environ["HF_TOKEN_API_KEY"]
)

In [9]:
%%time
embedded_documents = hf_embeddings.embed_documents(texts)

CPU times: total: 0 ns
Wall time: 4.91 s


In [10]:
print("[HuggingFace Endpoint Embedding]")
print(f"Model : \t\t{model_name}")
print(f"Dimension : \t{len(embedded_documents[0])}")

[HuggingFace Endpoint Embedding]
Model : 		intfloat/multilingual-e5-large-instruct
Dimension : 	1024


In [11]:
embedded_query = hf_embeddings.embed_query("Langchain에 대해서 알려주세요")
embedded_query

[0.013396107591688633,
 0.02308698184788227,
 -0.0033948447089642286,
 -0.021145163103938103,
 0.022823384031653404,
 -0.011206066235899925,
 -0.027452798560261726,
 0.03423108905553818,
 0.02346562221646309,
 -0.03224636986851692,
 0.02355221100151539,
 0.020252952352166176,
 -0.028615755960345268,
 -0.004158924799412489,
 -0.024074537679553032,
 -0.02265232987701893,
 -0.06238862872123718,
 0.008058683015406132,
 -0.02660878747701645,
 -0.011364714242517948,
 0.06040974706411362,
 0.010706160217523575,
 -0.015839021652936935,
 -0.022491544485092163,
 -0.014068697579205036,
 1.2135545830460615e-06,
 -0.020985830575227737,
 -0.04167867824435234,
 -0.001856140443123877,
 -0.03784443438053131,
 -0.0018047612393274903,
 0.014287794940173626,
 -0.02464158646762371,
 -0.05604258552193642,
 -0.013734794221818447,
 0.025947755202651024,
 0.054026782512664795,
 0.04319503903388977,
 -0.027979321777820587,
 0.05403994023799896,
 -0.013909214176237583,
 0.055481184273958206,
 0.02766076847910881

- 임베딩된 질문과 문서 간의 유사도 계산하기

In [12]:
import numpy as np

np.array(embedded_query) @ np.array(embedded_documents).T

array([0.82319326, 0.85968412, 0.85920334, 0.87956455, 0.75655574])

In [13]:
sorted_idx = (np.array(embedded_query) @ np.array(embedded_documents).T).argsort()[::-1]
sorted_idx

array([3, 1, 2, 0, 4])

In [14]:
print("[Query] LangChain 에 대해서 알려주세요.\n ========================")

for i, idx in enumerate(sorted_idx):
    print(f"[{i}] {texts[idx]}")
    print()

[Query] LangChain 에 대해서 알려주세요.
[0] LangChain은 초거대 언어모델로 애플리케이션을 구축하는 과정을 단순화합니다.

[1] LangChain simplifies the process of building applications with large language models

[2] 랭체인 한국어 튜토리얼은 LangChain의 공식 문서, cookbook 및 다양한 실용 예제를 바탕으로 하여 사용자가 LangChain을 더 쉽고 효과적으로 활용할 수 있도록 구성되어 있습니다. 

[3] 안녕, 만나서 반가워.

[4] Retrieval-Augmented Generation (RAG) is an effective technique for improving AI responses.



### HuggingFaceEmbeddings
- 로컬에 자신의 모델을 다운받아서 사용

In [18]:
from langchain_huggingface.embeddings import HuggingFaceEmbeddings

model_name = "intfloat/multilingual-e5-large-instruct"

hf_embeddings = HuggingFaceEmbeddings(
    model_name= model_name,
    model_kwargs= {"device" : "cpu"},
    encode_kwargs = {"normalize_embeddings" : True}
)

In [20]:
%%time
embedded_documents = hf_embeddings.embed_documents(texts)

CPU times: total: 5.52 s
Wall time: 579 ms


In [21]:
print(f"Model: \t\t{model_name}")
print(f"Dimension: \t{len(embedded_documents[0])}")

Model: 		intfloat/multilingual-e5-large-instruct
Dimension: 	1024


- BGE-M3 임베딩
- BAAI/bge-m3 모델 사용

In [23]:
model_name = "BAAI/bge-m3"
model_kwargs = {"device" : "cpu"}
encode_kwargs = {"normalize_embeddings" : True}
hf_embeddings = HuggingFaceEmbeddings(
    model_name= model_name,
    model_kwargs= model_kwargs,
    encode_kwargs= encode_kwargs
)

%time
embedded_documents = hf_embeddings.embed_documents(texts)

print(f"Model: \t\t{model_name}")
print(f"Dimension: \t{len(embedded_documents[0])}")

CPU times: total: 0 ns
Wall time: 0 ns
Model: 		BAAI/bge-m3
Dimension: 	1024


In [24]:
embedded_query = hf_embeddings.embed_query("LangChain에 대해서 알려주세요")
embedded_documents = hf_embeddings.embed_documents(texts)

np.array(embedded_query) @ np.array(embedded_documents).T

sorted_idx = (np.array(embedded_query) @ np.array(embedded_documents).T).argsort()[::-1]

print("[Query] LangChain에 대해서 알려주세요\n =======================")

for i, idx in enumerate(sorted_idx):
    print(f"[{i}] {texts[idx]}")
    print()

[Query] LangChain에 대해서 알려주세요
[0] LangChain simplifies the process of building applications with large language models

[1] LangChain은 초거대 언어모델로 애플리케이션을 구축하는 과정을 단순화합니다.

[2] 랭체인 한국어 튜토리얼은 LangChain의 공식 문서, cookbook 및 다양한 실용 예제를 바탕으로 하여 사용자가 LangChain을 더 쉽고 효과적으로 활용할 수 있도록 구성되어 있습니다. 

[3] 안녕, 만나서 반가워.

[4] Retrieval-Augmented Generation (RAG) is an effective technique for improving AI responses.



### FlagEmbedding 

In [ ]:
#!uv pip install FlagEmbedding

error: Failed to read `numpy==2.5.3`
  cause: Failed to read metadata from installed package `numpy==2.5.3`
  cause: failed to open file `D:\kingSJ\hanwha_0902\ex_0922\.venv\Lib\site-packages\numpy-2.5.3.dist-info\METADATA`: 지정된 파일을 찾을 수 없습니다. (os error 2)

hint: `numpy` (v2.5.3) was included because `flagembedding` (v1.4.2) depends on `transformers>=4.44.2, <6.0.0` (v4.46.3) which depends on `numpy>=1.17`


In [4]:
from FlagEmbedding import BGEM3FlagModel

bge_flagmodel = BGEM3FlagModel(
    "BAAI/bge-m3", use_fp16= False
)

bge_encoded = bge_flagmodel.encode(texts, return_dense= True)

ValueError: Unable to compare versions for tokenizers>=0.23.1,<0.24.0: need=0.23.1 found=None. This is unusual. Consider reinstalling tokenizers.

In [2]:
import transformers
print(transformers.__version__)

ValueError: Unable to compare versions for tokenizers>=0.23.1,<0.24.0: need=0.23.1 found=None. This is unusual. Consider reinstalling tokenizers.